# 🚀 XGBoost Model Training

This notebook trains the XGBoost model.

**Prerequisites**: Run `01_preprocessing.ipynb` first to generate preprocessed data.

In [1]:
import sys, os
import warnings
warnings.filterwarnings('ignore')
import pickle

# Ensure PROJECT_P is on the path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd

# Project modules
from src.config import *
from src.utils import get_logger

logger = get_logger('XGBoost')
print('✅ Imports successful')

✅ Imports successful


## 1. Load Preprocessed Data

In [2]:
# Load preprocessed data from file
preprocess_file = os.path.join(MODEL_DIR, 'preprocessed_data.pkl')

if not os.path.exists(preprocess_file):
    raise FileNotFoundError(f'Please run 01_preprocessing.ipynb first to generate {preprocess_file}')

with open(preprocess_file, 'rb') as f:
    prep = pickle.load(f)

X_train = prep['X_train']
X_val = prep['X_val']
X_test = prep['X_test']
y_train = prep['y_train']
y_val = prep['y_val']
y_test = prep['y_test']

print(f'✅ Preprocessed data loaded')
print(f'   Train: {X_train.shape}')
print(f'   Val:   {X_val.shape}')
print(f'   Test:  {X_test.shape}')

✅ Preprocessed data loaded
   Train: (124012, 94)
   Val:   (26575, 94)
   Test:  (26575, 94)


## 2. Train XGBoost

In [3]:
from src.training import train_xgboost
import importlib
import src.training

# Reload to get latest version
importlib.reload(src.training)
from src.training import train_xgboost

xgb_model, xgb_history = train_xgboost(X_train, y_train, X_val, y_val, use_smote=True)
print('\n✅ XGBoost training complete')

19:11:35 | Training             | INFO    | ==================================================
19:11:35 | Training             | INFO    |   🚀 Training XGBoost
19:11:35 | Training             | INFO    | ==================================================
19:11:35 | Training             | INFO    |   SMOTE Resampling:
19:11:35 | Training             | INFO    |     Before: 119,573 legit, 4,439 fraud (1:26.9)
19:11:35 | Training             | INFO    |     After:  119,573 legit, 59,786 fraud (1:2.0)
19:11:35 | Training             | INFO    |     Synthetic samples: 55,347
19:11:35 | Training             | INFO    | ⏳ Starting: XGBoost training
19:11:35 | Models               | INFO    |   🚀 XGBoost created (n_estimators=500, max_depth=8, scale_pos_weight=26.9)
19:11:53 | Training             | INFO    | ✅ Finished: XGBoost training (17.6s)
19:11:53 | Training             | INFO    |   Val F1-Score: 0.5155
19:11:53 | Training             | INFO    |   Val ROC-AUC:  0.9102
19:11:53 | Train


✅ XGBoost training complete


## 3. Evaluate on Test Set

## 3.5 Training Curves & Feature Importance

In [5]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Feature Importance
feature_importance = xgb_model.feature_importances_
feature_names = prep['feature_names'] if 'feature_names' in prep else [f'Feature {i}' for i in range(len(feature_importance))]

# Sort by importance
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(15)

axes[0].barh(importance_df['feature'], importance_df['importance'], color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Importance Score', fontweight='bold')
axes[0].set_title('XGBoost - Top 15 Most Important Features', fontweight='bold')
axes[0].invert_yaxis()

# 2. Model Statistics
try:
    booster = xgb_model.get_booster()
    n_trees = booster.num_boosted_rounds()
except:
    n_trees = xgb_model.get_params().get('n_estimators', 'Unknown')

stats_text = f"""
📊 XGBoost Model Statistics

🌳 Trees: {n_trees}
📏 Max Depth: {xgb_model.get_params()['max_depth']}
🎯 Learning Rate: {xgb_model.get_params()['learning_rate']}
⚖️ Scale Pos Weight: {xgb_model.get_params()['scale_pos_weight']:.2f}
📈 Features Used: {len(feature_names)}
⭐ Top Feature: {importance_df.iloc[0]['feature']}
"""

axes[1].text(0.5, 0.5, stats_text, 
             ha='center', va='center', fontsize=11, family='monospace',
             bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8),
             transform=axes[1].transAxes)
axes[1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, '03_xgb_importance_and_stats.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ XGBoost feature importance and model statistics saved')

✅ XGBoost feature importance and model statistics saved


In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Predictions
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Metrics
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
}

print('\n' + '='*50)
print('XGBoost - Test Set Performance')
print('='*50)
for metric, value in metrics.items():
    print(f'{metric:15s}: {value:.4f}')
print('='*50)


XGBoost - Test Set Performance
Accuracy       : 0.9595
Precision      : 0.4536
Recall         : 0.6477
F1-Score       : 0.5336
ROC-AUC        : 0.9118


## 4. Save Model and Results

In [6]:
# Save results (predictions only, not full model)
xgb_results = {
    'y_pred': y_pred,
    'y_pred_proba': y_pred_proba,
    'metrics': metrics,
    'history': xgb_history,
}

results_file = os.path.join(MODEL_DIR, 'xgboost_results.pkl')
with open(results_file, 'wb') as f:
    pickle.dump(xgb_results, f)

print(f'\n✅ Results saved to {results_file}')
print('\n📝 Next: Run 04_train_hgnn.ipynb')



✅ Results saved to /Users/aaronr/Desktop/PROJECT_P/models/xgboost_results.pkl

📝 Next: Run 04_train_hgnn.ipynb
